# Graph-Based Fragment Reassembly & Format Validation
## Forensic Reconstruction for Deleted XFS / Btrfs Fragments

This notebook demonstrates:
1. **Graph Construction**: Fragment Nodes + Multi-Component Edge Scoring
2. **Ordering Solver**: Beam Search DAG Traversal with Loop Prevention
3. **File Reconstruction**: Stitched Byte Streams & SHA-256 Hashes
4. **Format-Aware Validation**: Deep Structural Integrity Parsing (PDF, ZIP, PNG, ELF, SQLite, Scripts)

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

from src.reassembly.synthetic import run_reassembly_experiment, fragment_file
from src.reassembly.edge_scoring import build_complete_reassembly_graph, EdgeWeightConfig
from src.reassembly.ordering import order_fragments_beam_search
from src.reassembly.reconstruct import reconstruct_file_from_ordering
from src.validation.validator import validate_reconstructed_file
import pandas as pd

print("Reassembly & Validation Modules Imported Successfully!")

In [ ]:
# =============================================================================
# Run Controlled Synthetic Benchmark (3, 5, 10, 20 Fragments)
# =============================================================================
from scripts.run_reassembly_eval import create_sample_files

sample_files = create_sample_files()
results = run_reassembly_experiment(sample_files, fragment_counts=[3, 5, 10, 20], seed=42)

summary_rows = []
for count, s in results["summary"].items():
    summary_rows.append({
        "Fragment Count": count,
        "Exact Sequence Accuracy": f"{s['exact_seq_acc']*100:.1f}%",
        "Pairwise Adjacency Acc": f"{s['pairwise_adj_acc']*100:.1f}%",
        "Position Placement Acc": f"{s['position_acc']*100:.1f}%",
        "Byte Reconstruction Acc": f"{s['byte_acc']*100:.1f}%"
    })

df = pd.DataFrame(summary_rows)
display(df)

In [ ]:
# =============================================================================
# Format Validation on Reconstructed Data
# =============================================================================
for fmt, data in sample_files.items():
    report = validate_reconstructed_file(data, expected_format=fmt)
    print(f"[{ 'PASS' if report.is_valid else 'FAIL' }] Format: {fmt.upper():<6} | Valid: {report.is_valid} | Status: {report.status}")
    print(f"       SHA-256: {report.sha256[:32]}... | Size: {report.reconstructed_size_bytes}B")